# Multi Agent
## Supervisor Pattern
'중앙 관리자 에이전트가 하위 특화 에이전트에게 일을 시킴

> 관리자 Agent > 캘린더 agent / 이메일 agent

In [ ]:
from dotenv import load_dotenv
load_dotenv()

### Calendar Agent

In [ ]:
from langchain.tools import tool

@tool
def create_calendar_event(
    title: str,
    start_time: str,       # ISO format: "2024-01-15T14:00:00"
    end_time: str,         # ISO format: "2024-01-15T15:00:00"
    attendees: list[str],  # email addresses
    location: str = ""
) -> str:
    """Create a calendar event. Requires exact ISO datetime format."""
    # Stub: In practice, this would call Google Calendar API, Outlook API, etc.
    return f"Event created: {title} from {start_time} to {end_time} with {len(attendees)} attendees"


@tool
def get_available_time_slots(
    attendees: list[str],
    date: str,  # ISO format: "2024-01-15"
    duration_minutes: int
) -> list[str]:
    """Check calendar availability for given attendees on a specific date."""
    # Stub: In practice, this would query calendar APIs
    return ["09:00", "14:00", "16:00"]


from datetime import date

CALENDAR_AGENT_PROMPT = (
    f"Today's date is {date.today().isoformat()}. "
    "You are a calendar scheduling assistant. "
    "Parse natural language scheduling requests (e.g., 'next Tuesday at 2pm') "
    "into proper ISO datetime formats. "
    "Use get_available_time_slots to check availability when needed. "
    "If there is no suitable time slot, stop and confirm unavailability in your response. "
    "Use create_calendar_event to schedule events. "
    "Always confirm what was scheduled in your final response."
)

### Email Agent

In [ ]:
@tool
def send_email(
    to: list[str],  # email addresses
    subject: str,
    body: str,
    cc: list[str] = []
) -> str:
    """Send an email via email API. Requires properly formatted addresses."""
    # Stub: In practice, this would call SendGrid, Gmail API, etc.
    return f"Email sent to {', '.join(to)} - Subject: {subject}"

EMAIL_AGENT_PROMPT = (
    "You are an email assistant. "
    "Compose professional emails based on natural language requests. "
    "Extract recipient information and craft appropriate subject lines and body text. "
    "Use send_email to send the message. "
    "Always confirm what was sent in your final response."
)


In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

supervisor_llm = init_chat_model('openai:gpt-5.4-mini')
worker_llm = init_chat_model('openai:gpt-4.1-mini')

calenar_agent = create_agent(
    model=worker_llm,
    tools=[create_calendar_event, get_available_time_slots],
    system_prompt=CALENDAR_AGENT_PROMPT
)

email_agent = create_agent(
    model=worker_llm,
    tools=[send_email],
    system_prompt=EMAIL_AGENT_PROMPT
)

In [ ]:
from langchain.messages import HumanMessage

calenar_agent.invoke({
    'messages': [
        HumanMessage('내일중에 비어있는 시간에 David과 놀러가기 잡아줘')
    ]
})

In [12]:
email_agent.invoke({
    'messages': [
        HumanMessage('linda@nodecrew.com 에게 점심메뉴를 물어봐')
    ]
})

{'messages': [HumanMessage(content='linda@nodecrew.com 에게 점심메뉴를 물어봐', additional_kwargs={}, response_metadata={}, id='ef487662-851d-4139-8bd3-da1908793aed'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 127, 'total_tokens': 184, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_dc9dc22150', 'id': 'chatcmpl-EOY9CwvuKZKyZmPYpZ3Y1Mzu8xVWl', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0a7b1-cd17-7aa0-af49-65749676812c-0', tool_calls=[{'name': 'send_email', 'args': {'to': ['linda@nodecrew.com'], 'subject': '점심 메뉴 문의